In [4]:
# 1. Install dependencies
!apt-get update -qq
!apt-get install -y -qq zstd
!pip install -q langchain langchain-ollama requests

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start Ollama server
import subprocess
import time
import requests

subprocess.run(["pkill", "-9", "ollama"],
               stdout=subprocess.DEVNULL,
               stderr=subprocess.DEVNULL)

log_file = open("/tmp/ollama.log", "w")

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# 4. Wait for server
for i in range(30):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2)
        print("Ollama server started successfully!")
        break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError(
        "Ollama server failed to start:\n" +
        open("/tmp/ollama.log").read()
    )

# 5. Download Qwen3
subprocess.run(
    ["ollama", "pull", "qwen3:0.6b"],
    check=True
)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama server started successfully!


CompletedProcess(args=['ollama', 'pull', 'qwen3:0.6b'], returncode=0)

In [6]:
from pydantic.functional_serializers import model_serializer
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen3:0.6b",
    base_url="http://127.0.0.1:11434",
    temperature=0
)

# 7. Test model
response = model.invoke("Explain Artificial Intelligence in simple words.")
print(response.content)

Artificial Intelligence (AI) is like a special kind of computer that can do things that humans would normally do. It helps machines learn from data and make better decisions over time. For example, a computer might recognize patterns in images or text to help with tasks like medical diagnosis or traffic prediction. AI uses machines to learn from data, improving their ability to solve problems. It's a field that's growing really fast, helping with everything from everyday tasks to complex problems.


In [7]:
# Input
question = "What is University of Central Punjab?"

# Process using LangChain
response = model.invoke(question)

# Output
print(response.content)

The University of Central Punjab (UCP) is a public university located in Lahore, Pakistan. It was established in 1948 and is the largest university in the country. The university focuses on education, research, and providing quality education to the local population. It plays a significant role in Pakistan's education system and contributes to the development of the region.


In [8]:
# Input
question = "What is web?"

# Process using LangChain
response = model.invoke(question)

# Output
print(response.content)

The web is a vast network of interconnected systems and technologies that enable communication and information sharing. It consists of:

1. **Servers**: These are computers or devices that store and process data.
2. **Clients**: These are the computers or devices that access the web.
3. **Browsers**: These are programs that open websites.
4. **The Internet**: A global network of interconnected computers and servers.

The web allows people and organizations to connect, share information, and communicate across the globe. It's used in various fields like education, business, entertainment, and more.


In [9]:
import gradio as gr

In [10]:
# Chatbot function
def chatbot(question):
    response = model.invoke(question)
    return response.content

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(label="Enter your question"),
    outputs=gr.Textbox(label="AI Answer"),
    title="Qwen AI Chatbot"
)

# Launch chatbot
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://730bf716a1fd2f58f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
